In [0]:
def predict_and_evaluate(model_or_path, device, table_name: str, out_table: str = 'workspace.f1_racing_laptime_pred.gold_predicted_validation', label_threshold_sec: float = 1.0, target_mean: float = None, target_std: float = None, compute_pace_label: bool = False):
    """Run model predictions on Unity Catalog table, compute MAE and save predictions.

    - model_or_path: model object or path to state_dict. If path is provided, loads into module `model`.
    - table_name: Unity Catalog table name to read from (e.g., 'workspace.f1_racing_laptime_pred.silver_validating')
    - out_table: Unity Catalog table name to write predictions to
    - Returns: (mae, out_df)
    """
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    
    # Read from Unity Catalog instead of CSV
    print(f"Reading data from {table_name}...")
    df_pred = spark.table(table_name).toPandas()
    df_pred = assign_stints(df_pred)

    pred_dataset = StintDataset(df_pred, cont_features)
    pred_loader = DataLoader(pred_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

    preds_list = []
    actuals_list = []
    indices_list = []
    stint_counter = 0

    # load model if a path provided
    if isinstance(model_or_path, str):
        model.load_state_dict(torch.load(model_or_path, map_location=device))
        model.to(device)
        m = model
    else:
        m = model_or_path

    with torch.no_grad():
        for cont_feats, driver_idx, team_idx, tyre_idx, mode_idx, lap_time, mask in pred_loader:
            cont_feats = cont_feats.to(device)
            driver_idx = driver_idx.to(device)
            team_idx = team_idx.to(device)
            tyre_idx = tyre_idx.to(device)
            mode_idx = mode_idx.to(device)
            mask = mask.to(device)

            out = m(cont_feats, driver_idx, team_idx, tyre_idx, mode_idx, mask)
            out = out.cpu()

            for i in range(out.size(0)):
                valid_len = (~mask[i]).sum().item()
                preds_seq = out[i, :valid_len].tolist()
                actuals_seq = lap_time[i, :valid_len].tolist()

                stint_info = pred_dataset.stints[stint_counter]
                inds = stint_info.get('indices', list(range(valid_len)))[:valid_len]

                preds_list.extend(preds_seq)
                actuals_list.extend(actuals_seq)
                indices_list.extend(inds)
                stint_counter += 1

    res_df = pd.DataFrame({'orig_index': indices_list, 'pred': preds_list, 'actual': actuals_list})
    out_df = df_pred.reset_index().rename(columns={'index': 'orig_index'})
    out_df = out_df.merge(res_df, on='orig_index', how='left')

    # Prefer to compute MAE in seconds if possible (either via passed mean/std or saved scalers)
    mae_in_seconds = None
    if (target_mean is not None) and (target_std is not None):
        out_df['pred_sec'] = out_df['pred'] * target_std + target_mean
        out_df['actual_sec'] = out_df['actual'] * target_std + target_mean
        mae_in_seconds = mean_absolute_error(out_df['actual_sec'].dropna(), out_df['pred_sec'].dropna())
        print(f'MAE on {table_name}: {mae_in_seconds:.4f} seconds (rescaled using provided mean/std)')
    else:
        mae = mean_absolute_error(out_df['actual'].dropna(), out_df['pred'].dropna())
        print(f'MAE on {table_name}: {mae:.4f} (same units as training target)')

    # Compute residuals relative to seconds if available, otherwise in training units
    if 'pred_sec' in out_df.columns and 'actual_sec' in out_df.columns:
        out_df['residual_sec'] = out_df['actual_sec'] - out_df['pred_sec']
        if compute_pace_label:
            def label_row_seconds(r):
                if pd.isna(r):
                    return None
                if r <= -label_threshold_sec:
                    return 'pushing'
                elif r >= label_threshold_sec:
                    return 'conserving'
                else:
                    return 'doing_nothing'
            out_df['pace_label'] = out_df['residual_sec'].apply(label_row_seconds)
    else:
        out_df['residual'] = out_df['actual'] - out_df['pred']
        if compute_pace_label:
            def label_row_unit(r):
                if pd.isna(r):
                    return None
                if r <= -label_threshold_sec:
                    return 'pushing'
                elif r >= label_threshold_sec:
                    return 'conserving'
                else:
                    return 'doing_nothing'
            out_df['pace_label'] = out_df['residual'].apply(label_row_unit)
    
    # Write to Unity Catalog Gold layer
    print(f"Writing predictions to {out_table}...")
    spark_out_df = spark.createDataFrame(out_df)
    spark_out_df.write.mode("overwrite").saveAsTable(out_table)
    print(f'✓ Predictions written to {out_table}')
    
    if 'pace_label' in out_df.columns:
        print('Label counts:')
        print(out_df['pace_label'].value_counts(dropna=True))
    else:
        print('No `pace_label` column (set compute_pace_label=True to enable labeling).')

    return mae_in_seconds if mae_in_seconds is not None else mae, out_df

In [0]:
def calculate_prediction_errors(table_name='workspace.f1_racing_laptime_pred.gold_predicted_validation'):
    """
    Calculate error statistics from prediction results in Unity Catalog.
    
    Args:
        table_name: Unity Catalog table name containing predictions
    
    Returns:
        Dictionary containing error statistics
    """
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    
    # Load the predictions from Unity Catalog
    df = spark.table(table_name).toPandas()
    
    # Calculate error metrics
    errors = {
        'biggest_error': df['residual'].abs().max(),
        'smallest_error': df['residual'].abs().min(),
        'average_error': df['residual'].mean(),
        'average_absolute_error': df['residual'].abs().mean(),
        'mse': (df['residual'] ** 2).mean(),
        'rmse': np.sqrt((df['residual'] ** 2).mean()),
        'std_error': df['residual'].std(),
        'median_absolute_error': df['residual'].abs().median(),
    }
    
    # Error distribution
    distribution = {
        'total_predictions': len(df),
        'errors_gt_1sec': (df['residual'].abs() > 1).sum(),
        'errors_gt_2sec': (df['residual'].abs() > 2).sum(),
        'errors_gt_3sec': (df['residual'].abs() > 3).sum(),
    }
    
    return errors, distribution

In [0]:
def print_error_report(table_name='workspace.f1_racing_laptime_pred.gold_predicted_validation'):
    """
    Print a formatted report of prediction errors from Unity Catalog.
    """
    errors, distribution = calculate_prediction_errors(table_name)
    
    print("=" * 60)
    print("PREDICTION ERROR STATISTICS")
    print("=" * 60)
    
    print("\n--- Basic Error Metrics (in seconds) ---")
    print(f"Biggest error (absolute):        {errors['biggest_error']:.4f}")
    print(f"Smallest error (absolute):       {errors['smallest_error']:.4f}")
    print(f"Average error (mean residual):   {errors['average_error']:.4f}")
    print(f"Average absolute error (MAE):    {errors['average_absolute_error']:.4f}")
    print(f"Mean Squared Error (MSE):        {errors['mse']:.4f}")
    print(f"Root Mean Squared Error (RMSE):  {errors['rmse']:.4f}")
    
    print("\n--- Additional Statistics ---")
    print(f"Standard deviation of errors:    {errors['std_error']:.4f}")
    print(f"Median absolute error:           {errors['median_absolute_error']:.4f}")
    
    print("\n--- Error Distribution ---")
    total = distribution['total_predictions']
    print(f"Total predictions:               {total}")
    print(f"Errors > 1 second:               {distribution['errors_gt_1sec']:,} ({100*distribution['errors_gt_1sec']/total:.1f}%)")
    print(f"Errors > 2 seconds:              {distribution['errors_gt_2sec']:,} ({100*distribution['errors_gt_2sec']/total:.1f}%)")
    print(f"Errors > 3 seconds:              {distribution['errors_gt_3sec']:,} ({100*distribution['errors_gt_3sec']/total:.1f}%)")
    
    print("\n" + "=" * 60)
    
    return errors, distribution